# F7-kernels-convex-optimization — Practice p06

**Type:** constrained coding · **Difficulty:** core · **Concepts:** positive-semidefinite-matrices, kernel-validity

Implement `poly2_gram(X)` for the degree-two polynomial kernel

$$k(x,z)=(1+x^Tz)^2.$$

Input contract: `X` is a finite real-numeric NumPy array of shape `(n, d)` with `n,d >= 1`. Reject any other input with `ValueError`; do not mutate it. Return the finite floating NumPy array `K` of shape `(n,n)` with `K[i,j]=k(X[i],X[j])`.

Use `ATOL = 1e-10`, `RTOL = 0.0`. Your finite computation must agree with the formula, be symmetric, and be PSD within the declared tolerance. This checks the listed input only; the lesson's feature-map/closure argument is what proves validity for every finite list.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

def poly2_gram(X):
    """Build the Gram matrix for k(x, z) = (1 + x^T z)^2."""
    if (
        not isinstance(X, np.ndarray)
        or X.ndim != 2
        or X.shape[0] < 1
        or X.shape[1] < 1
        or not np.issubdtype(X.dtype, np.number)
        or np.iscomplexobj(X)
        or not np.isfinite(X).all()
    ):
        raise ValueError("X must be a finite real-numeric nonempty matrix")

    X_float = X.astype(float, copy=False)
    K = (1.0 + X_float @ X_float.T) ** 2
    if not np.isfinite(K).all():
        raise ValueError("the floating Gram computation must remain finite")
    return K

## Immutable contract check — do not edit

The public fixtures change both shape and values, compare every entry independently, and check type, finiteness, symmetry, PSD evidence, rejection, and non-mutation.

In [ ]:
_fixtures_p06 = (
    np.array([[2.0, -1.0]]),
    np.array([[1.0, 2.0], [-1.0, 0.5], [0.0, 0.0]]),
    np.array([[3.0], [-2.0], [0.25], [1.5]]),
    np.array([[0.1, -0.2, 0.3], [1.2, 0.0, -0.7]]),
)
for _X_p06 in _fixtures_p06:
    _before_p06 = _X_p06.copy()
    _K_p06 = poly2_gram(_X_p06)
    _expected_p06 = (1.0 + _X_p06 @ _X_p06.T) ** 2
    assert np.array_equal(_X_p06, _before_p06)
    assert isinstance(_K_p06, np.ndarray) and _K_p06.shape == (_X_p06.shape[0], _X_p06.shape[0])
    assert np.issubdtype(_K_p06.dtype, np.floating) and np.isfinite(_K_p06).all()
    assert np.allclose(_K_p06, _expected_p06, atol=ATOL, rtol=RTOL)
    assert np.allclose(_K_p06, _K_p06.T, atol=ATOL, rtol=RTOL)
    assert np.linalg.eigvalsh((_K_p06 + _K_p06.T) / 2.0)[0] >= -ATOL

_invalid_p06 = (
    [[1.0, 2.0]],
    np.array([1.0, 2.0]),
    np.empty((0, 2)),
    np.empty((2, 0)),
    np.array([[1.0, np.inf]]),
    np.array([[1.0 + 0.0j]]),
    np.array([["x"]]),
)
for _bad_p06 in _invalid_p06:
    try:
        poly2_gram(_bad_p06)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid X must raise ValueError")

### Solution reasoning

The matrix product between X and its transpose contains every rowwise inner product, so adding one and squaring elementwise implements the kernel formula for all pairs at once. Because the computation starts from floating data, the result has floating dtype. Symmetry follows from the symmetric inner-product matrix. Kernel validity is universal because
\[
(1+x^Tz)^2=1+2x^Tz+(x^Tz)^2
\]
is an inner product after combining constant, scaled-linear, and degree-two tensor features; the numerical eigenvalue check only confirms the supplied finite fixtures.

### Answer check

In [ ]:
_X_answer_p06 = np.array([[1.0, 2.0], [-1.0, 0.0]])
_K_answer_p06 = poly2_gram(_X_answer_p06)
assert np.allclose(
    _K_answer_p06,
    np.array([[36.0, 0.0], [0.0, 4.0]]),
    atol=ATOL,
    rtol=RTOL,
)
assert np.linalg.eigvalsh(_K_answer_p06)[0] >= -ATOL